# 05 — LCEL: Encadeando etapas com LangChain

Neste notebook vamos refazer o exemplo anterior de `SimpleSequentialChain`, mas utilizando **LCEL (LangChain Expression Language)**.

A ideia continua sendo a mesma: dividir uma tarefa em etapas e passar a saída de uma etapa para a próxima.

A diferença está na forma como construímos o fluxo.

### Abordagem anterior

```text
SimpleSequentialChain
    ↓
LLMChain
    ↓
LLMChain
```

### Abordagem com LCEL

```text
PromptTemplate
    ↓
LLM
    ↓
PromptTemplate
    ↓
LLM
```

O principal operador que vamos aprender é `|`, chamado de **pipe**.

## 1. O que é LCEL?

LCEL significa **LangChain Expression Language**.

Ela permite combinar componentes do LangChain utilizando uma sintaxe declarativa.

Por exemplo:

```python
chain = prompt | llm
```

Isso significa:

```text
entrada
   ↓
prompt
   ↓
llm
   ↓
resposta
```

A saída de um componente é passada automaticamente para o próximo.

## 2. Importando os componentes

Não precisamos mais importar `LLMChain` ou `SimpleSequentialChain`.

Vamos trabalhar diretamente com `ChatOllama` e `PromptTemplate`.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

## 3. Configurando o Qwen

Continuamos usando o Qwen local através do Ollama.

In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)

## 4. Criando o primeiro prompt

Nossa primeira etapa será explicar um conceito para um estudante iniciante.

In [ ]:
prompt_explicacao = PromptTemplate.from_template(
    "Explique o conceito de {conceito} para um estudante iniciante de tecnologia."
)

## 5. Criando a primeira chain com LCEL

Agora utilizamos o operador `|`:

In [ ]:
chain_explicacao = prompt_explicacao | llm

Podemos interpretar essa linha como:

```text
PromptTemplate → Qwen
```

Primeiro o template recebe os dados. Depois o prompt preenchido é enviado para o modelo.

## 6. Testando a primeira chain

Passamos o valor da variável `conceito` para a chain.

In [ ]:
resultado_1 = chain_explicacao.invoke({
    "conceito": "RAG"
})

print(resultado_1.content)

## 7. Criando o segundo prompt

A segunda etapa receberá a explicação produzida pela primeira etapa e deverá resumir essa explicação.

In [ ]:
prompt_resumo = PromptTemplate.from_template(
    "Resuma a explicação abaixo em uma frase simples:\n\n{explicacao}"
)

## 8. Criando a segunda chain

Também podemos criar uma chain simples para o segundo prompt:

In [ ]:
chain_resumo = prompt_resumo | llm

## 9. Encadeando as duas etapas

Agora chegamos ao ponto principal.

Queremos fazer:

```text
conceito
   ↓
chain_explicacao
   ↓
explicacao
   ↓
chain_resumo
   ↓
resumo
```

Como a segunda etapa espera uma variável chamada `explicacao`, precisamos transformar a saída da primeira chain em um dicionário com essa chave.

In [ ]:
chain = (
    {"explicacao": chain_explicacao}
    | prompt_resumo
    | llm
)

### Entendendo o dicionário

Esta parte é especialmente importante:

```python
{"explicacao": chain_explicacao}
```

A saída da `chain_explicacao` é associada à chave `explicacao`.

Isso permite que o próximo `PromptTemplate` encontre a variável:

```python
{explicacao}
```

Depois o resultado segue para o segundo `llm`.

## 10. Executando a chain completa

Agora fornecemos apenas a entrada inicial.

In [ ]:
resultado = chain.invoke({
    "conceito": "RAG"
})

## 11. Visualizando o resultado

In [ ]:
print(resultado.content)

## 12. Comparando com SimpleSequentialChain

No exercício anterior tínhamos algo parecido com:

```python
chain = SimpleSequentialChain(
    chains=[chain_explicacao, chain_resumo]
)
```

Com LCEL, construímos explicitamente o fluxo:

```python
chain = (
    {"explicacao": chain_explicacao}
    | prompt_resumo
    | llm
)
```

A grande diferença é que o LCEL nos dá mais controle sobre como os dados passam entre os componentes.

## 13. Um exemplo ainda mais simples

Antes de continuar, vale memorizar a ideia fundamental do LCEL.

Uma chain simples pode ser:

In [ ]:
chain_simples = prompt_explicacao | llm

Isso pode ser lido literalmente como:

```text
Prompt → Modelo
```

E podemos adicionar mais componentes:

```text
Entrada
  ↓
Prompt
  ↓
LLM
  ↓
Parser
```

Essa composição de componentes é uma das ideias centrais do LCEL.

## 14. Código completo

Agora juntamos o exemplo inteiro:

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)

prompt_explicacao = PromptTemplate.from_template(
    "Explique o conceito de {conceito} para um estudante iniciante de tecnologia."
)

prompt_resumo = PromptTemplate.from_template(
    "Resuma a explicação abaixo em uma frase simples:\n\n{explicacao}"
)

chain_explicacao = prompt_explicacao | llm

chain = (
    {"explicacao": chain_explicacao}
    | prompt_resumo
    | llm
)

resultado = chain.invoke({
    "conceito": "RAG"
})

print(resultado.content)